# 🧪 موتور آزمایشی — بازطراحی با کنترل عمق (Depth ControlNet + Juggernaut XL v9)

این یک نوت‌بوک **کاملاً جدا و آزمایشی**ه — به موتور فعلی پروژه (Colab، SD1.5/SDXL) هیچ کاری نداره و جایگزینش نمی‌کنه. فقط برای تست کیفیت یه رویکرد متفاوته: به‌جای لبه‌یابی Canny، از **نقشهٔ عمق (Depth Map)** برای حفظ ساختار اتاق و زاویهٔ دوربین استفاده می‌کنه، با یه مدل SDXL دیگه (Juggernaut XL v9) که مخصوص فوتورئالیسمه.

## مشخصات فنی
| | |
|---|---|
| مدل تولید | `RunDiffusion/Juggernaut-XL-v9` |
| کنترل ساختار | `diffusers/controlnet-depth-sdxl-1.0` |
| استخراج عمق | `Intel/dpt-hybrid-midas` |
| Pipeline | `StableDiffusionXLControlNetImg2ImgPipeline` |
| محیط | Kaggle، ۱ GPU از ۲ تا T4 (۱۵ گیگ VRAM) |

## قوانین این نوت‌بوک
- فقط یک GPU استفاده می‌شه (`cuda:0`)
- همه‌چیز fp16، فقط فایل‌های لازم دانلود می‌شن (نه کل مخزن)
- مدل عمق اول اجرا می‌شه، عمق استخراج می‌شه، بعد **حافظه‌اش آزاد می‌شه**، بعد مدل تولید لود می‌شه
- هر اجرا یک عکس تولید می‌کنه، نسبت ابعاد عکس ورودی حفظ می‌شه
- زمان تولید، اوج مصرف VRAM و seed هر بار چاپ و ذخیره می‌شه
- بدون IP-Adapter، SAM، مدل زبانی یا آپ‌اسکیلر — فقط هستهٔ اصلی

## قبل از اجرا
یه عکس واقعی از یه اتاق آپلود کن (Kaggle → Add Data → Upload، یا مستقیم تو `/kaggle/working/`) و مسیرش رو تو سلول «تنظیمات» بذار.


## ۱) بررسی محیط
نسخهٔ فعلی PyTorch/CUDA رو فقط چاپ می‌کنیم — عمداً تغییرش نمی‌دیم.

In [ ]:
import os, torch

# Only GPU 0 of the two T4s — per the spec, start with a single GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (torch build):", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

DEVICE = "cuda:0"
DTYPE = torch.float16

## ۲) نصب کتابخانه‌ها
**فقط** `diffusers`/`transformers`/`accelerate`/`safetensors` — به‌عمد `torch` رو دوباره نصب نمی‌کنیم تا نسخهٔ از‌پیش‌نصب‌شده و سازگار با CUDA کگل دست‌نخورده بمونه.

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers>=0.31.0", "transformers", "accelerate", "safetensors", "huggingface_hub"
])
print("✅ Libraries installed (torch untouched)")

## ۳) تنظیمات — مسیر عکس ورودی و پرامپت رو اینجا عوض کن

In [ ]:
from PIL import Image

# ─── مسیر عکس واقعی اتاق رو اینجا بذار ───
INPUT_IMAGE_PATH = "/kaggle/working/room.jpg"   # عوضش کن به مسیر عکس خودت

# توصیف دکوراسیون جدید — ساختار اتاق (دیوار، پنجره، زاویه) حفظ می‌شه، فقط جنس/رنگ/مبلمان عوض می‌شه
PROMPT = (
    "modern scandinavian interior design, warm oak wood tones, soft neutral palette, "
    "natural daylight, cozy textiles, minimalist decor, photorealistic, interior photography, "
    "8k, highly detailed, architectural digest style"
)
NEGATIVE_PROMPT = (
    "different room layout, moved walls, moved windows, changed camera angle, "
    "blurry, low quality, distorted, watermark, text, oversaturated, cartoon"
)

SEED = None  # None = یه seed رندوم انتخاب و لاگ می‌شه؛ برای تکرارپذیری یه عدد بذار (مثلاً 42)

original_image = Image.open(INPUT_IMAGE_PATH).convert("RGB")
print("Input size:", original_image.size)

## ۴) استخراج نقشهٔ عمق
مدل `Intel/dpt-hybrid-midas` رو لود می‌کنیم، عمق رو استخراج می‌کنیم، بعد **حافظه‌اش رو کامل آزاد می‌کنیم** — قبل از اینکه مدل تولید (که خودش چند گیگ دیگه جا می‌خواد) لود بشه.

In [ ]:
import gc
import numpy as np
import torch.nn.functional as F
from transformers import DPTImageProcessor, DPTForDepthEstimation


def resize_to_multiple_of_8(w, h, max_side=1024):
    """Preserve the input aspect ratio, capped at max_side on the long edge,
    rounded to a multiple of 8 (SDXL requirement)."""
    scale = max_side / max(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    new_w -= new_w % 8
    new_h -= new_h % 8
    return max(new_w, 8), max(new_h, 8)


GEN_W, GEN_H = resize_to_multiple_of_8(*original_image.size)
print(f"Generation size (aspect-preserved): {GEN_W}x{GEN_H}")

_depth_processor = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
_depth_model = DPTForDepthEstimation.from_pretrained(
    "Intel/dpt-hybrid-midas", torch_dtype=torch.float32  # small model, fp32 is fine and avoids precision issues
).to(DEVICE)
_depth_model.eval()

_inputs = _depth_processor(images=original_image, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    _pred = _depth_model(**_inputs).predicted_depth

_depth = F.interpolate(
    _pred.unsqueeze(1), size=(GEN_H, GEN_W), mode="bicubic", align_corners=False
)
_dmin, _dmax = torch.amin(_depth), torch.amax(_depth)
_depth = (_depth - _dmin) / (_dmax - _dmin + 1e-8)
_depth_rgb = torch.cat([_depth] * 3, dim=1)[0].permute(1, 2, 0).cpu().numpy()
depth_map = Image.fromarray((_depth_rgb * 255.0).clip(0, 255).astype(np.uint8))

# ── Free the depth model before loading the (much bigger) generation pipeline ──
del _depth_model, _depth_processor, _inputs, _pred, _depth, _depth_rgb
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Depth map extracted ({depth_map.size}); depth model unloaded, VRAM freed")
print("VRAM allocated right now (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

## ۵) لود موتور تولید
قبل از دانلود، چک می‌کنیم که آیا نسخهٔ fp16 هر مخزن موجوده یا نه — اگه بود، فقط همونو می‌گیریم (نه fp32 رو هم اضافه).

In [ ]:
from huggingface_hub import list_repo_files
from diffusers import ControlNetModel, AutoencoderKL, StableDiffusionXLControlNetImg2ImgPipeline


def has_fp16_variant(repo_id):
    files = list_repo_files(repo_id)
    return any("fp16" in f for f in files)


CONTROLNET_ID = "diffusers/controlnet-depth-sdxl-1.0"
BASE_MODEL_ID = "RunDiffusion/Juggernaut-XL-v9"
VAE_ID = "madebyollin/sdxl-vae-fp16-fix"  # SDXL's default VAE NaNs in fp16 — this fixed one is the standard fix

controlnet_kwargs = dict(torch_dtype=DTYPE, use_safetensors=True)
if has_fp16_variant(CONTROLNET_ID):
    controlnet_kwargs["variant"] = "fp16"
controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, **controlnet_kwargs)

vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=DTYPE)

base_kwargs = dict(controlnet=controlnet, vae=vae, torch_dtype=DTYPE, use_safetensors=True)
if has_fp16_variant(BASE_MODEL_ID):
    base_kwargs["variant"] = "fp16"

pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(BASE_MODEL_ID, **base_kwargs)
pipe = pipe.to(DEVICE)
# Juggernaut-XL (~7GB fp16) + depth ControlNet (~2.5GB fp16) comfortably fit a
# 15GB T4 on their own — kept fully on GPU (no offload) for speed. If you hit
# an OOM, the fallback is: pipe.enable_model_cpu_offload() instead of .to(DEVICE).
pipe.enable_vae_tiling()

print("✅ Generation pipeline ready on", DEVICE)
print("VRAM allocated right now (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

## ۶) تولید — یک عکس، با ثبت زمان/VRAM/seed

In [ ]:
import time

seed = SEED if SEED is not None else torch.seed() % (2**31)
generator = torch.Generator(device=DEVICE).manual_seed(seed)

torch.cuda.reset_peak_memory_stats(DEVICE)
_t0 = time.time()

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=original_image.resize((GEN_W, GEN_H)),
    control_image=depth_map,
    height=GEN_H,
    width=GEN_W,
    strength=0.55,                     # how much to deviate from the original photo (img2img)
    controlnet_conditioning_scale=0.8,  # how strongly to follow the depth map
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
    num_images_per_prompt=1,
).images[0]

_elapsed = time.time() - _t0
_peak_vram_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9

print(f"✅ Done in {_elapsed:.1f}s")
print(f"Peak VRAM: {_peak_vram_gb:.2f} GB")
print(f"Seed used: {seed}")

result.save("/kaggle/working/result.jpg", quality=95)
depth_map.save("/kaggle/working/depth_map.jpg", quality=90)

RUN_LOG = {
    "model": BASE_MODEL_ID,
    "controlnet": CONTROLNET_ID,
    "depth_model": "Intel/dpt-hybrid-midas",
    "seed": seed,
    "elapsed_seconds": round(_elapsed, 2),
    "peak_vram_gb": round(_peak_vram_gb, 2),
    "generation_size": [GEN_W, GEN_H],
    "strength": 0.55,
    "controlnet_conditioning_scale": 0.8,
    "num_inference_steps": 30,
    "guidance_scale": 7.5,
}
import json as _json
with open("/kaggle/working/run_log.json", "w") as f:
    _json.dump(RUN_LOG, f, indent=2, ensure_ascii=False)
print(_json.dumps(RUN_LOG, indent=2, ensure_ascii=False))

## ۷) مقایسهٔ نتیجه

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original_image.resize((GEN_W, GEN_H)))
axes[0].set_title("Original")
axes[1].imshow(depth_map)
axes[1].set_title("Depth map (control_image)")
axes[2].imshow(result)
axes[2].set_title("Result")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## ۸) بعد از بررسی

این نوت‌بوک عمداً **جدا** نگه داشته شده. بعد از اینکه چند بار با عکس‌های واقعی امتحانش کردی و از سه چیز مطمئن شدی:

1. **کیفیت** — نتیجه واقعاً از موتور فعلی بهتره؟
2. **حفظ ساختار** — دیوار/پنجره/زاویهٔ دوربین واقعاً ثابت می‌مونه؟ (مقدار `strength` و `controlnet_conditioning_scale` رو لازمه تیون کنی — این دو عدد شروع، نه نهایی‌ان)
3. **سرعت** — با توجه به `elapsed_seconds` که لاگ می‌شه، برای استفادهٔ واقعی به‌صرفه‌ست؟

...اون‌وقت با خبرم کن تا این موتور رو (به‌عنوان یه گزینهٔ سوم کنار SD1.5/SDXL فعلی) به بک‌اند اصلی وصل کنیم — دقیقاً مثل کاری که برای مدل Fast/Quality کردیم.
